# 1. Step 1: Khởi tạo Spark Session và Nạp dữ liệu
Tiến hành khởi tạo môi trường Spark và nạp dữ liệu Parquet đã được làm sạch từ quá trình ETL.

In [ ]:
# Nạp thư viện và khởi tạo dữ liệu
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

def initialize_and_load_data():
    # Khởi tạo Spark Session cho Machine Learning
    spark = SparkSession.builder \
        .appName("MovieRecommendation-ALS") \
        .master("local[*]") \
        .getOrCreate()
        
    # Nạp dữ liệu Ratings từ thư mục data
    rating_path = "../../data/processed_ratings.parquet"
    ratings_df = spark.read.parquet(rating_path)
    
    return spark, ratings_df

spark_session, ratings_data = initialize_and_load_data()
print("Đã nạp thành công dữ liệu để huấn luyện AI.")

# 2. Step 2: Tiền xử lý (Chia tập Train/Test)
Chia dữ liệu thành 2 tập: 80% để huấn luyện (Train) và 20% để kiểm thử độ chính xác (Test).

In [ ]:
# Hàm chia cắt tập dữ liệu (80% Train, 20% Test)
def split_data(df):
    # Sử dụng randomSplit với seed=42 để đảm bảo tính nhất quán mỗi lần chạy
    train, test = df.randomSplit([0.8, 0.2], seed=42)
    return train, test

training_data, test_data = split_data(ratings_data)
print(f"Số lượng bản ghi Train: {training_data.count()}")
print(f"Số lượng bản ghi Test: {test_data.count()}")

# 3. Step 3: Huấn luyện Mô hình ALS
Sử dụng thuật toán Alternating Least Squares (ALS) của thư viện Spark MLlib để học đặc trưng và thói quen của người dùng.

In [ ]:
# Hàm khởi tạo và huấn luyện mô hình Trí tuệ Nhân tạo
def train_als_model(train_df):
    # Cấu hình thuật toán ALS (Alternating Least Squares)
    als_algo = ALS(maxIter=10, 
                   regParam=0.1, 
                   userCol="user_id", 
                   itemCol="movie_id", 
                   ratingCol="rating",
                   coldStartStrategy="drop")
    
    # Thực thi việc học thói quen người dùng
    trained_model = als_algo.fit(train_df)
    return trained_model

print("Bắt đầu huấn luyện mô hình ALS...")
als_model = train_als_model(training_data)
print("Huấn luyện hoàn tất.")

# 4. Step 4: Đánh giá độ chính xác (Tính điểm RMSE)
Sử dụng hàm RegressionEvaluator để kiểm tra xem mô hình dự đoán lệch bao nhiêu điểm so với thực tế (RMSE - Root Mean Square Error).

In [ ]:
# Hàm chấm điểm sai số (RMSE) của mô hình
def evaluate_model(model, test_df):
    # Bảo AI làm bài kiểm tra trên tập Test
    predictions = model.transform(test_df)
    
    # Tính toán sai số lệch chuẩn
    evaluator = RegressionEvaluator(metricName="rmse", 
                                    labelCol="rating",
                                    predictionCol="prediction")
    
    error_rate = evaluator.evaluate(predictions)
    return predictions, error_rate

predictions_df, rmse_score = evaluate_model(als_model, test_data)

print(f"ĐIỂM SAI SỐ RMSE CỦA MÔ HÌNH: {rmse_score:.4f}")
predictions_df.select("user_id", "movie_id", "rating", "prediction").show(5)

# 5. Step 5 & 6: Dự đoán toàn hệ thống và Xuất kết quả
Mô hình sẽ tự động tính toán và dự đoán Top 10 bộ phim hay nhất cho TẤT CẢ khách hàng. Sau đó lưu mảng dữ liệu này thành định dạng Parquet để Backend (MongoDB) sử dụng.

In [ ]:
# Hàm sinh gợi ý phim và xuất dữ liệu ra file
def generate_and_save_recommendations(model):
    # Sinh ra Top 10 bộ phim hay nhất cho toàn bộ hệ thống
    recommendations = model.recommendForAllUsers(10)
    
    # Xuất ra thư mục data dưới dạng Parquet để Backend lấy dùng
    output_path = "../../data/user_recommendations.parquet"
    recommendations.write.mode("overwrite").parquet(output_path)
    
    return recommendations

print("Đang tiến hành dự đoán và xuất file...")
final_recommendations = generate_and_save_recommendations(als_model)
print("Hoàn tất! Danh sách gợi ý đã sẵn sàng cho Giao diện Web.")

# Xem thử cấu trúc mảng 10 phim của một khách hàng
final_recommendations.show(5, truncate=False)